# Week 3 · Part 1 — From the mean to a regression line

**Question for today:** can we predict the price of a flat from how far it is from the nearest metro station?

This is a **regression** task: the answer is a number (a price), not a category. In Week 2 you predicted a category (penguin species); today the target is continuous.

### Plan
1. Meet the data
2. Describe the target: mean, median, variance, standard deviation, distribution
3. Look at one relationship
4. Split the rows into training and test sets
5. Build the simplest possible model: always predict the mean
6. Measure error with MAE
7. Fit a regression line with least squares
8. Check the line against the lecture formulas
9. Compare the line with the baseline on test rows
10. Read residuals
11. What the slope means, and what it does not

### Words you will meet
| Word | Meaning |
|---|---|
| target | the column we want to predict |
| input (feature) | a column the model uses to predict |
| baseline | a very simple model that any real model must beat |
| residual | actual value − predicted value |
| MAE | mean absolute error: the average size of a miss |
| least squares | the rule that chooses the line with the smallest sum of squared residuals |

## 1. Meet the data
The dataset contains 414 real property sales in Sindian District, New Taipei City, Taiwan. **One row is one sale.**

| Column | Meaning |
|---|---|
| `record_id` | Row number from the original file. A label, **not** an input. |
| `transaction_date_decimal` | Date of sale as a decimal year: 2013.250 is March 2013. Sales run from August 2012 to July 2013. |
| `house_age_years` | Age of the building in years. |
| `distance_to_mrt_m` | Distance to the nearest MRT (metro) station, in metres. |
| `convenience_stores` | Number of convenience stores within walking distance. |
| `latitude`, `longitude` | Location in degrees. |
| `price_10k_twd_per_ping` | **Target.** Price per unit area, in 10,000 New Taiwan dollars per Ping. One Ping is about 3.3 m². A value of 38 means 380,000 TWD per Ping. |

First we load the tools. Each import is one tool we will use today; the comment says what it is for.

In [ ]:
import numpy as np                                    # numerical calculations
import pandas as pd                                   # tables (DataFrames)
import matplotlib.pyplot as plt                       # plots
from sklearn.model_selection import train_test_split  # split rows into training and test sets
from sklearn.dummy import DummyRegressor              # baseline model that ignores the inputs
from sklearn.linear_model import LinearRegression     # least-squares regression line
from sklearn.metrics import mean_absolute_error       # average size of prediction errors

In [ ]:
df = pd.read_csv("Dataset.csv")
target = "price_10k_twd_per_ping"
print("rows, columns:", df.shape)
df.head()

**What we see:** 414 rows and 8 columns. The last column is the target. The six columns between `record_id` and the target are possible inputs. Today we use only one of them: `distance_to_mrt_m`.

## 2. Describe the target
Before building any model we describe the target. This tells us what a "large" error is.

| Statistic | Meaning | Lecture 3 |
|---|---|---|
| **mean** | sum of all values ÷ number of values | slide 39 |
| **median** | the middle value after sorting | slide 40 |
| **variance** s² | average squared distance from the mean, dividing by n − 1. Its unit is price **squared**, which is hard to read. | slide 44 |
| **standard deviation** s | √variance. Back in price units: a typical distance from the mean. | slide 43 |

In [ ]:
price = df[target]
print("mean:              ", round(price.mean(), 3))
print("median:            ", round(price.median(), 3))
print("variance:          ", round(price.var(), 3))
print("standard deviation:", round(price.std(), 3))
print("min / max:         ", price.min(), "/", price.max())

**What we see:** the average price is about 38.0 and the middle price is 38.45, so they are close. The standard deviation is 13.6: a typical sale is about 13.6 units away from the mean. The variance, 185.1, is the same information in squared units. The maximum, 117.5, is far above everything else.

Now the same variance and standard deviation **by the formula**, to see that pandas does exactly what the lecture describes. Slide 42 also says the deviations from the mean always add up to zero; we check that too.

In [ ]:
n = len(price)
deviations = price - price.mean()                    # how far each price is from the mean
variance_by_hand = (deviations ** 2).sum() / (n - 1)
sd_by_hand = np.sqrt(variance_by_hand)
print("variance by hand:", round(variance_by_hand, 3), "  SD by hand:", round(sd_by_hand, 3))
print("deviations sum to zero:", bool(np.isclose(deviations.sum(), 0)))

**What we see:** the same numbers as pandas. pandas divides by n − 1 by default, like the lecture.

### The distribution of prices
A **distribution** shows how often each range of values occurs. A histogram is a picture of it: the height of each bar is the number of sales in that price range.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(price, bins=25, edgecolor="white")
ax.axvline(price.mean(), color="#D1495B", linewidth=2, label="mean")
ax.axvline(price.median(), color="#00798C", linewidth=2, label="median")
ax.set(title="Distribution of price per unit area", xlabel="10,000 TWD per Ping", ylabel="number of sales")
ax.legend(); plt.show()

**What we see:** most prices (82%) lie between 20 and 55, with one peak near the mean. The red mean line and the green median line almost overlap because the two values differ by only 0.5. A long thin tail runs to the right, up to 117.5.

For a bell-shaped (normal) distribution, about 68% of values lie within one standard deviation of the mean and about 95% within two. Let us check how close our prices are to that rule.

In [ ]:
within_1 = (deviations.abs() <= price.std()).mean()
within_2 = (deviations.abs() <= 2 * price.std()).mean()
print("share within 1 SD of the mean:", round(within_1, 3))
print("share within 2 SD of the mean:", round(within_2, 3))

**What we see:** 67% and 98%, close to 68% and 95%. Apart from the long tail, the prices behave roughly like a bell-shaped distribution. We will meet the normal distribution again in Part 3, when we look at errors.

## 3. Look at one relationship
Before fitting a model, look at the data. In a scatter plot each point is one sale: distance on the horizontal axis, price on the vertical axis.

In [ ]:
feature = "distance_to_mrt_m"
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df[feature], df[target], alpha=.6)
ax.set(title="Price and distance to the nearest MRT station",
       xlabel="distance (metres)", ylabel="price (10,000 TWD per Ping)")
plt.show()

**What we see:**
- Sales close to a station (left) are usually more expensive; distant ones (right) are cheaper.
- The fall is steep at first and then flattens: the cloud is **curved**, not straight.
- At the same distance, prices still vary a lot, because other things matter too.

We start with a straight line because it is the simplest model. The curve will return in Part 3.

A plot shows **association** in this sample. It does not prove that moving a flat closer to a station would make it more expensive.

## 4. Split the rows into training and test sets
As in Week 2, the model learns from **training rows** and is judged on **test rows** it has never seen. This is the only honest way to estimate how well it predicts new sales.

`train_test_split(df, test_size=0.20, random_state=42)` puts a random 20% of rows into the test set. `random_state=42` fixes the random shuffle, so everyone gets exactly the same rows.

In [ ]:
train, test = train_test_split(df, test_size=0.20, random_state=42)
print("training rows:", len(train), "  test rows:", len(test))
print("rows in both sets:", len(set(train.index) & set(test.index)))

**What we see:** 331 training rows, 83 test rows, and no row in both.

scikit-learn models expect the inputs as a **table** (`X`) and the target as **one column of numbers** (`y`). Double brackets `train[[feature]]` keep a table with one column; single brackets `train[target]` give a single column.

In [ ]:
X_train, X_test = train[[feature]], test[[feature]]
y_train, y_test = train[target], test[target]
print("X_train shape:", X_train.shape, "  y_train shape:", y_train.shape)

**What we see:** `X_train` has 331 rows and 1 column; `y_train` has 331 values.

## 5. The simplest model: always predict the mean
Before a clever model, ask: **how well can we do without using any input?** If we must give one number for every flat, the natural choice is the average training price.

**New tool — `DummyRegressor(strategy="mean")`.** It is the regression version of the `DummyClassifier` from Week 2.
- `.fit(X_train, y_train)` learns only one thing: the mean of `y_train`. It ignores `X_train`.
- `.predict(X_test)` returns that same number for every row.

We use the **training** mean, not the mean of all rows, because the test prices must stay unseen.

In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
print("training mean:         ", round(y_train.mean(), 3))
print("first five predictions:", dummy_pred[:5].round(3))

**What we see:** the dummy predicts 38.392 for every flat, whatever its distance.

## 6. Measure error with MAE
For each test flat, the **error** is actual price − predicted price. Some errors are positive (we predicted too low) and some negative (too high). If we simply average them, they cancel out. So we average their **absolute values**:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

MAE is in the same unit as the price, so it reads directly: "on average, the prediction misses by MAE units." First by hand for the dummy:

In [ ]:
check = pd.DataFrame({"actual": y_test, "predicted": dummy_pred})
check["error"] = check["actual"] - check["predicted"]
check["absolute error"] = check["error"].abs()
display(check.head().round(3))
print("mean of errors:         ", round(check["error"].mean(), 3))
print("mean of absolute errors:", round(check["absolute error"].mean(), 3))

**What we see:** the plain mean of errors is only −2.05 because large positive and negative errors cancel. The mean of absolute errors, 10.917, is the real typical miss.

**New tool — `mean_absolute_error(actual, predicted)`** computes the same number in one call:

In [ ]:
dummy_mae = mean_absolute_error(y_test, dummy_pred)
print("dummy MAE:", round(dummy_mae, 3))

**What we see:** 10.917, the same as by hand. The baseline misses by about 10.9 units, or 109,000 TWD per Ping. **Any useful model must beat this number on the same test rows.**

## 7. Fit a regression line with least squares
A simple linear regression predicts with a straight line (Lecture 3, slides 14–16):

$$\hat{y} = b_0 + b_1 x$$

- $b_0$, the **intercept**: the predicted price when distance is 0.
- $b_1$, the **slope**: how much the predicted price changes for one more metre.

**Least squares** (slides 17–18) chooses $b_0$ and $b_1$ so that the sum of squared training residuals $\sum (y_i - \hat{y}_i)^2$ is as small as possible.

**New tool — `LinearRegression()`.**
- `.fit(X_train, y_train)` finds $b_0$ and $b_1$ by least squares, using training rows only.
- It stores them as `.intercept_` and `.coef_` (one coefficient per input).
- `.predict(X)` applies the formula to new rows.

In [ ]:
line = LinearRegression()
line.fit(X_train, y_train)
b0 = line.intercept_
b1 = line.coef_[0]
print("intercept b0:", round(b0, 4))
print("slope b1:    ", round(b1, 6))

**What we see:** the fitted line is

**predicted price = 46.243 − 0.00741 × distance in metres**

- Each extra 100 metres lowers the predicted price by 0.741 units.
- At distance 0 the line predicts 46.2. The nearest sale is 23 m from a station, so this intercept is close to real data and makes sense here.

`predict` only applies this formula. Check it for a flat 500 m from a station:

In [ ]:
distance = 500
by_hand = b0 + b1 * distance
by_model = line.predict(pd.DataFrame({feature: [distance]}))[0]
print("by hand: ", round(by_hand, 3))
print("by model:", round(by_model, 3))

**What we see:** both give 42.538.

## 8. Check the line against the lecture formulas
Slides 19–20 give the least-squares solution directly:

$$b_1 = \frac{S_{xy}}{S_{xx}}, \qquad b_0 = \bar{y} - b_1\bar{x}$$

where $S_{xy} = \sum (x_i - \bar{x})(y_i - \bar{y})$ and $S_{xx} = \sum (x_i - \bar{x})^2$. We compute them from the **training** rows.

In [ ]:
x, y = X_train[feature], y_train
S_xy = ((x - x.mean()) * (y - y.mean())).sum()
S_xx = ((x - x.mean()) ** 2).sum()
b1_formula = S_xy / S_xx
b0_formula = y.mean() - b1_formula * x.mean()
print("formula b0:", round(b0_formula, 4), "  formula b1:", round(b1_formula, 6))
print("same as LinearRegression:", bool(np.isclose(b0_formula, b0) and np.isclose(b1_formula, b1)))

**What we see:** exactly the same coefficients. `LinearRegression` does the least-squares calculation from the lecture.

Two more facts from the lecture:
- With an intercept, least-squares **training residuals always sum to zero**.
- The error variance is estimated as $\hat{\sigma}^2 = \frac{\sum r_i^2}{n-2}$ (slide 26). We divide by n − 2 because two numbers, $b_0$ and $b_1$, were estimated from the same rows.

In [ ]:
train_residuals = y - (b0 + b1 * x)
print("training residuals sum to zero:", bool(np.isclose(train_residuals.sum(), 0)))
sigma2 = (train_residuals ** 2).sum() / (len(x) - 2)
print("residual variance estimate:", round(sigma2, 3), "  residual SD:", round(np.sqrt(sigma2), 3))

**What we see:** the residuals sum to zero, and the residual SD is 10.4. Without the line, prices spread around their mean with SD 13.6; around the line they spread with SD 10.4. The line explains part of the variation, not all of it.

## 9. Compare the line with the baseline on test rows
Now the fair comparison: **same test rows, same metric**.

In [ ]:
line_pred = line.predict(X_test)
line_mae = mean_absolute_error(y_test, line_pred)
print("dummy MAE:  ", round(dummy_mae, 3))
print("line MAE:   ", round(line_mae, 3))
print("improvement:", round(dummy_mae - line_mae, 3), f"({(dummy_mae - line_mae) / dummy_mae:.0%} smaller)")

**What we see:** the line misses by 6.92 on average, against 10.92 for the baseline: an error about 37% smaller. Distance carries real information about price.

## 10. Read residuals
A **residual** is actual − predicted.
- **Positive** residual: the actual price is higher than predicted. The model predicted **too low**.
- **Negative** residual: the actual price is lower than predicted. The model predicted **too high**.

In the plot, each grey vertical segment is the residual of one test flat: the gap between the point and the line.

In [ ]:
grid = pd.DataFrame({feature: np.linspace(df[feature].min(), df[feature].max(), 200)})
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.vlines(X_test[feature], line_pred, y_test, color="grey", alpha=.6, label="residual")
ax.scatter(X_test[feature], y_test, s=18, label="test sales (actual)")
ax.plot(grid[feature], line.predict(grid), color="#D1495B", linewidth=2.5, label="fitted line")
ax.axhline(0, color="black", linewidth=.8)
ax.set(title="Fitted line and test residuals", xlabel="distance (metres)", ylabel="price (10,000 TWD per Ping)")
ax.legend(); plt.show()

**What we see:**
- Near stations the points spread widely above and below the line.
- Between about 1,000 and 2,500 m most points lie **below** the line: there the line predicts too high.
- Beyond 4,000 m most points lie **above** the line, and the line itself goes **below zero** at the right edge, although a price can never be negative.

A straight line is too simple for this curved relationship. We return to this in Parts 2 and 3.

The same residuals as a table:

In [ ]:
results = test[["record_id", feature, target]].copy()
results["predicted"] = line_pred
results["residual"] = results[target] - results["predicted"]
results.head(8).round(3)

**What we see:**
- Record 374 sold for 52.2; the line predicted 44.2. Residual +8.0: predicted **too low**.
- Record 400 sold for 37.3; the line predicted 45.0. Residual −7.7: predicted **too high**.
- Records 263 and 141 have exactly the same distance, 289.325 m, and the same prediction, but different prices. Several sales share a location, and distance alone cannot separate them.

Which test flat received the lowest prediction?

In [ ]:
results.loc[results["predicted"].idxmin()].round(2)

**What we see:** record 348, 6,488 m from a station, sold for 11.2, but the line predicted −1.83. The farthest training sale is 6,306 m away, so here the line is used **outside the range it learned from**. This is called **extrapolation**. A straight line keeps falling forever; real prices do not.

## 11. What the slope means, and what it does not
- **Means:** in these 331 training sales, flats 100 m farther from a station were on average about 0.74 units cheaper, according to the fitted line.
- **Does not mean:** moving a flat 100 m closer would raise its price by 0.74. Distant flats may also be older, larger, or in different neighbourhoods. The slope is an **association**, not a cause.
- **Limits:** one district, sales from 2012–2013, only one input, a relationship that is curved rather than straight, and extrapolation at large distances.

### What you learned
- Describe the target first: mean, median, variance, standard deviation, distribution.
- A baseline (`DummyRegressor`) gives the number to beat.
- MAE is the average absolute miss, in the target's unit.
- `LinearRegression` finds the least-squares line, identical to the lecture formulas.
- Residuals show where and how the model misses.

In **Practice 3A** you repeat these steps with a different input.

**Sources:** AITU Lecture 3, slides 14–20, 26, 28–34, and 39–44; ISLP Chapter 3, physical PDF pages 79–86; UCI Real Estate Valuation, DOI 10.24432/C5J30W (CC BY 4.0).